# Principal Components Analysis

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/PCATutorial.m`](../matlab/PCATutorial.m) by Fred Rieke.

This tutorial introduces **principal components analysis** (PCA). The aims are:

1. introduce **dimensional reduction** and why it matters;
2. introduce the **covariance** and the **covariance matrix**;
3. show how the **eigensystem** of the covariance matrix ranks the dimensions of a
   probability distribution by how much variance each captures;
4. illustrate all of this with a real application — the trial-to-trial variability of
   **single-photon responses in rod photoreceptors**.

It builds on two ideas introduced earlier in the course: probability distributions and
eigensystems. If either feels shaky, take another pass through the relevant sections of the
stochastic-processes and linear-algebra tutorials first.

| Part | Topic |
|---|---|
| I | Motivation: why we want to reduce dimensions |
| II | Covariance and the covariance matrix |
| III | The eigensystem of the covariance matrix |
| IV | Application: principal components of rod single-photon responses |

A companion tutorial, [`PCANeuroPopTutorial.ipynb`](PCANeuroPopTutorial.ipynb), applies the
same machinery to the *geometry* of neural population activity. This one is about
**variability**: characterizing how a set of responses differs from trial to trial. Read
them in either order; the covariance/eigensystem core is shared, the applications are not.

Run the notebook cell by cell (**Shift+Enter**), and read the text — the narrative and the
homework problems are the tutorial, not decoration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg
from scipy.io import loadmat
from sklearn.decomposition import PCA

rng = np.random.default_rng(545)   # fixed seed, so every run reproduces the figures below

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 11, "font.size": 9,
})

BLUE, RED, GREEN, GREY = "#3366d9", "#d94d3f", "#33914f", "#808080"

DATA = "../matlab"     # relative to python/ in the course repo

### A note on the data files

Both `.mat` files here are old (2003 and 2007) MATLAB **5.0** format files, so
`scipy.io.loadmat` reads them directly — no `h5py`, and no `squeeze_me` /
`struct_as_record` gymnastics, because neither file contains structs or cell arrays. Each
holds plain numeric matrices:

| file | variable | shape | MATLAB dtype |
|---|---|---|---|
| `rgc-spike-response.mat` | `RGCSpikes` | (21, 4000) | `uint8` — 0/1 spike indicator, 1 ms bins |
| `RodData.mat` | `Singles` | (236, 120) | `double` — single-photon responses, 10 ms samples |
| `RodData.mat` | `Failures` | (348, 120) | `double` — dark records, same sampling |

Two things still need unwrapping, and both bite silently:

- **Byte order.** These files were written on PowerPC Macs, so the doubles come back with
  dtype `>f8` (big-endian). NumPy handles that correctly, but the non-native dtype
  propagates into everything you compute and some libraries choke on it. Cast once with
  `.astype(float)` on load.
- **Integer type.** `RGCSpikes` is `uint8`. `RGCSpikes.mean(axis=0)` is fine, but anything
  that subtracts (a mean-subtraction, a difference of trials) would **wrap around** rather
  than go negative. Cast to `float` on load.

`loadmat` also returns three bookkeeping keys — `__header__`, `__version__`, `__globals__` —
which is why the loading cell below filters on the names it actually wants.

In [ ]:
rgc = loadmat(f"{DATA}/rgc-spike-response.mat")
rod = loadmat(f"{DATA}/RodData.mat")

print("rgc-spike-response.mat:", [k for k in rgc if not k.startswith("__")])
print("RodData.mat           :", [k for k in rod if not k.startswith("__")])
print()
print("raw dtypes as loaded  :", rgc["RGCSpikes"].dtype, rod["Singles"].dtype)

RGCSpikes = rgc["RGCSpikes"].astype(float)   # (trials, 4000) 0/1, 1 ms bins
Singles   = rod["Singles"].astype(float)     # (236, 120) single-photon responses
Failures  = rod["Failures"].astype(float)    # (348, 120) dark records

for name, A in [("RGCSpikes", RGCSpikes), ("Singles", Singles), ("Failures", Failures)]:
    print(f"{name:10s} {A.shape}  {A.dtype}  range [{A.min():.3g}, {A.max():.3g}]")

---
## Part I. Motivation for dimensional reduction

What do we mean by dimensional reduction, and why should you care?

With finite experimental data we always face a tradeoff: identify all the important
structure in the data, while not being misled by randomness that comes from having a finite
number of samples. The classic example is a **post-stimulus time histogram** (PSTH) of spike
responses to a repeated stimulus. In choosing the histogram bin width you trade smoothing
over real temporal structure against inventing spurious structure out of finite data.

Here is a set of spike responses from a retinal ganglion cell to a dim flash of light
delivered at $t = 0$. Each row is one trial; the recording runs from $-1$ to $3$ s in 1 ms
bins.

In [ ]:
dt = 0.001
t_rgc = np.arange(-1, 3, dt)          # 4000 points, flash at t = 0
n_show = 10

fig, ax = plt.subplots(figsize=(9, 4))
for resp in range(n_show):
    spike_times = t_rgc[RGCSpikes[resp] > 0]
    ax.vlines(spike_times, resp + 0.6, resp + 1.4, color="k", lw=0.8)
ax.axvline(0, color=RED, lw=1.2, ls="--")
ax.set(xlabel="time (s)", ylabel="trial", title=f"RGC spike responses to a dim flash "
       f"({n_show} of {RGCSpikes.shape[0]} trials)",
       xlim=(-1, 3), ylim=(0.3, n_show + 0.7), yticks=range(1, n_show + 1))
ax.grid(False)
fig.tight_layout()

The MATLAB original drew this with `plot(RGCSpikes(resp,:)/1.2 + resp)`, which paints a
line at every one of the 4000 bins and puts a spike on top of it. A raster of tick marks
(`ax.vlines`) shows the same data and is far easier to read.

Now the PSTH. The data are already discretized into 1 ms bins, so some information about
exact spike times has been thrown away — and yet a PSTH at that resolution still looks
terrible.

In [ ]:
# Rebinning by block-averaging. The MATLAB original used decimate(x, 100, 1), an order-1
# Chebyshev lowpass followed by downsampling; decimating by a factor of 100 in one step
# that way produces filter ringing and is not what anyone means by "100 ms bins". A block
# mean is what the text describes, and it is exactly a boxcar filter plus downsampling.
ResampleFactor = 100                                  # 100 ms bins
n_trials, n_bins = RGCSpikes.shape
Resampled = RGCSpikes.reshape(n_trials, n_bins // ResampleFactor, ResampleFactor).mean(2)
t_coarse = t_rgc[::ResampleFactor] + 0.5 * ResampleFactor * dt   # bin centers

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=False)
axes[0].plot(t_rgc, RGCSpikes.mean(0), color=BLUE, lw=0.6)
axes[0].set(title=f"1 ms bins ({n_bins} dimensions)", xlabel="time (s)",
            ylabel="spike probability per bin", xlim=(-1, 3))
axes[1].plot(t_coarse, Resampled.mean(0), color=BLUE, lw=1.6, marker="o", ms=3)
axes[1].set(title=f"100 ms bins ({Resampled.shape[1]} dimensions)", xlabel="time (s)",
            ylabel="spike probability per 1 ms bin", xlim=(-1, 3))
for ax in axes:
    ax.axvline(0, color=RED, lw=1, ls="--")
fig.tight_layout()

print(f"{n_trials} trials, {RGCSpikes.sum():.0f} spikes total "
      f"({RGCSpikes.sum() / n_trials:.1f} per trial)")

In the left panel you can at least see an extra density of spikes just after the flash, but
you also sense that most of the wiggle is noise. In effect we are keeping too many
variables — one per 1 ms time bin. That is the same as representing each spike train as a
point in a **4000-dimensional space**, with one axis per bin.

The right panel keeps only 40 dimensions and looks far better. But now we might worry that
we have smoothed away real structure. Part of this problem is unavoidable: with finite data
there is only so much structure you can separate from sampling error. What we would like is
to make the *best possible* use of the finite data — to choose the low-dimensional space
carefully so that the structure of the data is captured as effectively as possible.

You face this every time you run an experiment. By choosing a filter setting and a sampling
rate you are choosing what parts of the data you consider important.

PCA is about finding a low-dimensional representation of a data set **systematically**
rather than by an ad hoc rule. Smoothing until the PSTH looks nice is arbitrary and is not
guided by the structure of the responses themselves. PCA instead provides a set of
components — think of them as a set of axes in the space the data lives in — *and* tells us
how much of the structure in the data each axis captures. That should become concrete
below.

---
## Part II. Covariance and the covariance matrix

Consider a signal described by two variables, $x$ and $y$, with many samples of $(x, y)$
pairs. The **covariance matrix** is

$$\mathrm{Cov} = \begin{pmatrix}
\langle xx\rangle - \langle x\rangle\langle x\rangle &
\langle xy\rangle - \langle x\rangle\langle y\rangle \\
\langle xy\rangle - \langle x\rangle\langle y\rangle &
\langle yy\rangle - \langle y\rangle\langle y\rangle \end{pmatrix}$$

where $\langle x \rangle$ is the average of $x$ across samples. The diagonal elements are
the variances of $x$ (upper left) and $y$ (lower right). The off-diagonal terms are the
**covariance** of $x$ and $y$ — how much they change together. Each element is corrected for
what you would expect from the means alone: we take $\langle xy \rangle$ and subtract
$\langle x\rangle\langle y\rangle$, the value it would have if $x$ and $y$ were independent.

This is closely related to correlation coefficients. The correlation between $x$ and $y$ is

$$r_{xy} = \frac{\langle xy\rangle - \langle x\rangle\langle y\rangle}
{\sqrt{\mathrm{Var}(x)\,\mathrm{Var}(y)}},
\qquad\text{i.e.}\qquad
\mathrm{Cor}(i,j) = \frac{\mathrm{Cov}(i,j)}{\sqrt{\mathrm{Cov}(i,i)\,\mathrm{Cov}(j,j)}}$$

so the correlation matrix is the covariance matrix with each element normalized by the
square root of the product of the corresponding diagonal terms. Equivalently
$\mathrm{Cov}(i,j) = \mathrm{Cor}(i,j)\sqrt{\sigma_i^2\sigma_j^2}$.

### ⚠ The single most important MATLAB → NumPy gotcha in this tutorial

MATLAB's `cov(X)` treats **rows as observations and columns as variables**. NumPy's
`np.cov(X)` does the **opposite** by default — rows are variables. You must write

```python
np.cov(X, rowvar=False)      # rows = samples, columns = variables  (MATLAB's convention)
```

This is nasty precisely because it fails *silently*. With 10000 samples of 2 variables,
`np.cov(Dist)` returns a 10000×10000 matrix and you notice immediately. But feed it a
square block — say 120 responses of 120 time points — and you get a 120×120 matrix of
plausible-looking garbage with no warning at all. Every `np.cov` call in this notebook
passes `rowvar=False`; check yours.

### Example 1: $x$ and $y$ independent

Each row of `Dist` is one $(x, y)$ sample; column 0 is $x$, column 1 is $y$.

In [ ]:
n = 10000
Dist1 = np.column_stack([rng.normal(0, 0.3, n),    # x: normal, sd 0.3
                         rng.normal(0, 1.0, n)])   # y: normal, sd 1

# --- Example 2 (used in the same figure): x and y correlated ---
x2 = rng.normal(0, 0.3, n)
Dist2 = np.column_stack([x2, rng.normal(0, 1.0, n) + 1.5 * x2])

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, D, title in [(axes[0], Dist1, "Example 1: independent"),
                     (axes[1], Dist2, "Example 2: correlated")]:
    ax.plot(D[:, 0], D[:, 1], ".", ms=1.5, color=BLUE, alpha=0.4)
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", xlabel="x", ylabel="y", title=title)
fig.tight_layout()

In [ ]:
def covar_by_hand(D):
    '''Covariance matrix built element by element from <xx> - <x><x>, exactly as written
    in the text above. Uses the 1/N ("population") convention.'''
    x, y = D[:, 0], D[:, 1]
    varx    = np.mean(x * x) - np.mean(x) ** 2
    vary    = np.mean(y * y) - np.mean(y) ** 2
    covarxy = np.mean(x * y) - np.mean(x) * np.mean(y)
    return np.array([[varx, covarxy], [covarxy, vary]])

np.set_printoptions(precision=4, suppress=True)

print("Example 1 (independent)")
print("  by hand:\n", covar_by_hand(Dist1))
print("  np.cov(rowvar=False):\n", np.cov(Dist1, rowvar=False))
print("  correlation matrix:\n", np.corrcoef(Dist1, rowvar=False))
print()
print("Example 2 (correlated)")
print("  by hand:\n", covar_by_hand(Dist2))
print("  np.cov(rowvar=False):\n", np.cov(Dist2, rowvar=False))
print("  correlation matrix:\n", np.corrcoef(Dist2, rowvar=False))

For Example 1 the diagonal holds the variances of $x$ ($\approx 0.3^2 = 0.09$) and $y$
($\approx 1$), and the off-diagonal elements are small — nonzero only because our sample is
finite. (Change the seed and rerun to convince yourself.) The correlation matrix shows the
same structure, nicely normalized: ones on the diagonal by construction, near zero off it.

This lack of covariance is what the picture already told us. The spread along $x$ is
unrelated to the spread along $y$: if I told you the $x$ value you could not predict $y$.

For Example 2 the cloud is **tilted** in the $x$–$y$ plane, and correspondingly the
covariance matrix has substantial off-diagonal elements. Note also that the variance of $y$
grew: $y$ now contains $1.5x$ on top of its own unit-variance noise, so
$\mathrm{Var}(y) \approx 1 + 1.5^2(0.3^2) = 1.2$.

You can think of many cases where $x$ and $y$ are correlated. Two cells with common
synaptic input have correlated spike trains: let $x$ be the firing rate of one and $y$ the
firing rate of the other, and when $x$ runs high so does $y$. In that case knowing $x$ gives
you some ability to predict $y$ — which is exactly what a nonzero off-diagonal element
means.

### Sphering (whitening) a distribution

Here is another use of the covariance matrix. Suppose we want to take `Dist2` and
"spherize" it — this is what happens when you let the statisticians take over the language,
you get verbs like *to sphere*. We want new axes $x'$ and $y'$ in which the two variables
are uncorrelated and the standard deviation along each axis is 1.

We can do it with the inverse of the **matrix square root** of the covariance matrix. Why
the square root? The entries of the covariance are variances, so to get the units right we
need something more like a standard deviation.

Start with the pure scaling case from the linear algebra tutorial: a distribution already
aligned with the axes.

In [ ]:
# Note the layout change: here each COLUMN is a sample, so that M @ Dist works as written
# (this matches the MATLAB original, which switched conventions at this point too).
DistS = np.vstack([rng.normal(0, 0.5, 5000),
                   rng.normal(0, 1.5, 5000)])          # (2, 5000)

M = np.array([[0.5, 0.0],       # matrix carrying the sd along each axis
              [0.0, 1.5]])
NewDist = np.linalg.inv(M) @ DistS                     # rescale by the inverse of M

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, D, xl, yl, title in [
        (axes[0], DistS,   "x", "y", "original (sd 0.5, 1.5)"),
        (axes[1], NewDist, r"$x/\sigma_x$", r"$y/\sigma_y$", r"after $M^{-1}$")]:
    ax.plot(D[0], D[1], ".", ms=1.5, color=BLUE, alpha=0.4)
    ax.set(xlim=(-5, 5), ylim=(-5, 5), aspect="equal", xlabel=xl, ylabel=yl, title=title)
fig.tight_layout()

print("sd before:", DistS.std(1).round(3), "   sd after:", NewDist.std(1).round(3))

Now the sd along each axis is 1, the distribution is symmetric, and moving a distance 1
along either axis moves you 1 standard deviation.

Our real problem has an extra complication: the distribution is not lined up with the axes.
Inspired by the scaling matrix above, try the inverse of the square root of the covariance
matrix.

### ⚠ `Covar^0.5` in MATLAB is *not* `Covar**0.5` in NumPy

MATLAB's `^` on a matrix is the **matrix** power, so `Covar^0.5` is the matrix square root:
the matrix $A$ with $AA = \mathrm{Covar}$. NumPy's `**` on an ndarray is **elementwise**, so
`Covar**0.5` takes the square root of each entry and gives you something completely
different (and, for a covariance with negative off-diagonal terms, full of `nan`). Use
`scipy.linalg.sqrtm`, or `np.linalg.matrix_power` for integer powers.

In [ ]:
Covar2 = np.cov(Dist2, rowvar=False)

sqrtC = scipy.linalg.sqrtm(Covar2).real        # MATLAB Covar^0.5, NOT Covar**0.5
Dist2w = Dist2 @ np.linalg.inv(sqrtC)          # "spherized" / whitened

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, D, xl, yl, title in [
        (axes[0], Dist2,  "x",  "y",  "correlated distribution"),
        (axes[1], Dist2w, "x'", "y'", r"after $\mathrm{Cov}^{-1/2}$: spherized")]:
    ax.plot(D[:, 0], D[:, 1], ".", ms=1.5, color=BLUE, alpha=0.4)
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", xlabel=xl, ylabel=yl, title=title)
fig.tight_layout()

print("check sqrtm:  max|sqrtC @ sqrtC - Covar| =",
      np.abs(sqrtC @ sqrtC - Covar2).max())
print("covariance before whitening:\n", Covar2)
print("covariance after whitening:\n", np.cov(Dist2w, rowvar=False))

The inverse square root of the covariance has **rotated and scaled** the axes. In the
$x', y'$ coordinate system the distribution is circular — it has been spherized — and its
covariance matrix has come out as the identity to within sampling error.

### Higher dimensions

The covariance generalizes directly. Two more examples, now in six dimensions.

In [ ]:
NumDim = 6

# Example 3: 6-D, all components independent, sd of component d equal to d
Dist3 = np.column_stack([rng.normal(0, d, n) for d in range(1, NumDim + 1)])

# Example 4: 6-D, each component gets a contribution from the one before it
Dist4 = np.column_stack([rng.normal(0, d, n) for d in range(1, NumDim + 1)])
for d in range(1, NumDim):
    Dist4[:, d] = Dist4[:, d] + 0.5 * Dist4[:, d - 1]

print("Example 3: independent (expected diagonal 1, 4, 9, 16, 25, 36)")
print(np.cov(Dist3, rowvar=False))
print()
print("Example 4: neighbouring components coupled")
print(np.cov(Dist4, rowvar=False))

In [ ]:
# Same two matrices, drawn. Shared color scale, so the panels are genuinely comparable.
C3, C4 = np.cov(Dist3, rowvar=False), np.cov(Dist4, rowvar=False)
vmax = max(np.abs(C3).max(), np.abs(C4).max())

fig, axes = plt.subplots(1, 2, figsize=(9, 4.0))
for ax, Cm, title in [(axes[0], C3, "Example 3: independent"),
                      (axes[1], C4, "Example 4: coupled neighbours")]:
    im = ax.imshow(Cm, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    # The diagonal dominates the color scale, so print the numbers on top: the point is
    # WHICH off-diagonal elements are nonzero, not how big the diagonal is.
    for i in range(NumDim):
        for j in range(NumDim):
            ax.text(j, i, f"{Cm[i, j]:.1f}", ha="center", va="center", fontsize=7,
                    color="k" if abs(Cm[i, j]) < 0.6 * vmax else "w")
    ax.set(title=title, xlabel="component j", ylabel="component i",
           xticks=range(NumDim), yticks=range(NumDim))
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.85, label="covariance")

In Example 3 the covariance matrix is diagonal, with the diagonal equal to the variances
$1, 4, 9, 16, 25, 36$; the off-diagonal elements are not systematically different from zero.
In Example 4, introducing correlations between the variables produces nonzero off-diagonal
components — exactly as in the simple 2-D case. Look carefully at *which* off-diagonal
elements are large, and at whether any of them are zero; that is homework question 1(b).

Keep this picture of the covariance matrix in mind when we turn to the neuroscience
application below.

> ### Homework question 1
> **(a)** Explain the value of each component of the 2-D covariance matrix in Example 2
> above. You know how `Dist2` was constructed — predict all four numbers analytically and
> compare to what the code printed.
>
> **(b)** Which of the off-diagonal elements in the covariance matrix for Example 4 are
> nonzero? Why? Careful: component 3 was built from component 2, which was built from
> component 1 — so is $\mathrm{Cov}(1,3)$ zero?
>
> **(c)** Can you separate $\mathrm{Cov}^{-1/2}$ as used above into the product of a
> rotation matrix and a scaling matrix? *Hint:* you will know how to do this by the end of
> Part III.

---
## Part III. The eigensystem of the covariance matrix

The eigenvectors of the covariance matrix turn out to provide a very useful coordinate
system. In particular, **each eigenvalue is the variance of the distribution along the
associated eigenvector**. That is the whole of PCA in one sentence; the rest is bookkeeping.

Recall that eigenvectors $v$ satisfy $\mathrm{C}v = \lambda v$. Review the linear algebra
tutorial if that is unfamiliar.

### ⚠ Three eigen-gotchas

- **Use `np.linalg.eigh`, not `eig`, for a covariance matrix.** A covariance matrix is real
  and symmetric, so `eigh` applies; it is faster, more accurate, and returns *real*
  eigenvalues and orthonormal eigenvectors. `np.linalg.eig` on a symmetric matrix can return
  a tiny imaginary part and non-orthogonal vectors.
- **Ordering.** `eigh` returns eigenvalues in **ascending** order — smallest first. MATLAB's
  `eig` makes no ordering guarantee at all, though in practice it also came out ascending
  here, which is why the MATLAB original talks about the "last few" components. We will
  **sort descending** so that component 1 is always the largest. Watch for this when you
  compare the two files.
- **Sign is arbitrary.** If $v$ is an eigenvector, so is $-v$. A plotted eigenvector or
  principal component may come out mirrored relative to someone else's figure, or relative
  to the same code on a different machine, and nothing is wrong. Below we fix the sign by a
  stated convention so the figures are reproducible.

As with `eigh`, `np.linalg.eigvalsh` gives just the eigenvalues when you do not need the
vectors.

In [ ]:
def pca_eig(Cm):
    '''Eigen-decomposition of a symmetric (covariance) matrix, sorted DESCENDING by
    eigenvalue. Returns (eigvals, eigvecs) with eigvecs[:, k] the k-th eigenvector.

    Sign convention: flip each eigenvector so its largest-magnitude entry is positive.
    Eigenvector sign is mathematically arbitrary; fixing it makes figures reproducible.
    '''
    w, V = np.linalg.eigh(Cm)              # ascending, real, orthonormal
    order = np.argsort(w)[::-1]            # -> descending
    w, V = w[order], V[:, order]
    flip = np.sign(V[np.abs(V).argmax(axis=0), np.arange(V.shape[1])])
    flip[flip == 0] = 1.0
    return w, V * flip

In [ ]:
EigVal1, EigVec1 = pca_eig(np.cov(Dist1, rowvar=False))   # independent
EigVal2, EigVec2 = pca_eig(np.cov(Dist2, rowvar=False))   # correlated

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, D, w, V, title in [
        (axes[0], Dist1, EigVal1, EigVec1, "Example 1: independent"),
        (axes[1], Dist2, EigVal2, EigVec2, "Example 2: correlated")]:
    ax.plot(D[:, 0], D[:, 1], ".", ms=1.5, color=GREY, alpha=0.35)
    for k, color in [(0, RED), (1, GREEN)]:
        # Scale each eigenvector by 2 sqrt(lambda) -- i.e. 2 sd along that direction --
        # so the arrow length shows how much variance the axis actually carries.
        v = V[:, k] * 2 * np.sqrt(w[k])
        ax.annotate("", xy=v, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2))
        ax.plot([], [], color=color, lw=2.2,
                label=f"eigvec {k+1}, $\\lambda$ = {w[k]:.3f}")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", xlabel="x", ylabel="y", title=title)
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()

print("Example 1 eigenvalues:", EigVal1.round(4), " (variances of x, y were 0.09 and 1)")
print("Example 1 eigenvectors (columns):\n", EigVec1)
print("Example 2 eigenvalues:", EigVal2.round(4))
print("Example 2 eigenvectors (columns):\n", EigVec2)

In Example 1 the eigenvectors point along the $y$ and $x$ axes — they should be close to
$(0,1)$ and $(1,0)$ — which makes sense because the covariance matrix is nearly diagonal.
The eigenvalues are just the variances along those axes, $\approx 1$ and $\approx 0.09$.
(The MATLAB original's arrows come out in the other order because it did not sort; ours are
sorted largest-first, so the *first* eigenvector is the $y$ axis here.)

In Example 2 the eigenvectors no longer point along $x$ and $y$. The eigenvector with the
largest eigenvalue points along the direction in which the cloud of points is most extended,
and the second is orthogonal to it.

Why do the eigenvectors have this property? Recall from Part II that we could think of the
covariance matrix as a rotation combined with a scaling. The rotation part defines a new
coordinate system $x', y'$ in which the covariance is diagonal — the independent case. Seen
in *that* coordinate system, the eigenvectors line up with the axes. Seen in the original
coordinate system, they sit at whatever angle the distribution is tilted to.

Note that the arrows are drawn with length $2\sqrt{\lambda}$ — two standard deviations along
each direction — rather than unit length as in the MATLAB original's `PlotVector`. A unit
arrow tells you the direction but hides the ranking, which is the point of the whole
exercise.

### Checking the claim: eigenvalue = variance of the projection

Each row of `Dist` is an $(x,y)$ pair, i.e. the components of the vector pointing at that
data point. So the projections onto an eigenvector are just `Dist @ EigVec[:, k]`.

In [ ]:
Proj1 = Dist2 @ EigVec2[:, 0]
Proj2 = Dist2 @ EigVec2[:, 1]

print(f"var(projection onto eigvec 1) = {Proj1.var(ddof=1):.6f}   "
      f"eigenvalue 1 = {EigVal2[0]:.6f}")
print(f"var(projection onto eigvec 2) = {Proj2.var(ddof=1):.6f}   "
      f"eigenvalue 2 = {EigVal2[1]:.6f}")
print(f"\ncorrelation between the two projections: "
      f"{np.corrcoef(Proj1, Proj2)[0, 1]:.2e}  (uncorrelated by construction)")
print(f"sum of eigenvalues = {EigVal2.sum():.6f}   "
      f"trace of covariance = {np.trace(np.cov(Dist2, rowvar=False)):.6f}")

The variance of the projection equals the eigenvalue, and the two projections are
uncorrelated — the eigenvectors have *diagonalized* the covariance. Note also that the sum
of the eigenvalues equals the trace of the covariance matrix, which is the total variance.
That identity is what lets us talk about the "fraction of variance explained" by a subset of
components.

This all extends to higher dimensions.

In [ ]:
C4 = np.cov(Dist4, rowvar=False)
w4, V4 = pca_eig(C4)

print("eigenvalues (descending):", w4.round(3))
print()
for k in range(NumDim):
    proj = Dist4 @ V4[:, k]
    print(f"eigenvector {k+1}: var(projection) = {proj.var(ddof=1):9.4f}   "
          f"eigenvalue = {w4[k]:9.4f}")
print()
print(f"sum of eigenvalues {w4.sum():.4f} = trace of covariance {np.trace(C4):.4f}")
print("fraction of total variance per component:",
      (w4 / w4.sum()).round(4))
print("cumulative:", np.cumsum(w4 / w4.sum()).round(4))

The variance along each eigenvector is indeed equal to the corresponding eigenvalue. The
leading eigenvector carries about half of the total variance and the top three carry
roughly 88% of it, so even here — where the six variables were built with quite different
variances and only nearest-neighbour coupling — half the dimensions can be dropped at a
cost of about 10% of the variance.

Why is all this useful? The eigenvectors provide a coordinate system for the space the data
lives in — but not an arbitrary one. It is a coordinate system in which **each axis is
ranked by how much variance is associated with it**. That ranking is what gives us a
systematic way to reduce the number of dimensions: we can make a principled choice of which
axes to keep and which to discard, and we know exactly how much of the variance we capture
and how much we throw away.

---
## Part IV. Application: principal components of rod single-photon responses

This section shows how these tools give a low-dimensional description of **single-photon
responses in rod photoreceptors**. Rod responses to single photons vary from one to the
next, and we would like a compact way to characterize that variation.

The data set contains two matrices. Each row is one recorded trace, sampled every 10 ms:

- `Singles` (236 × 120): single-photon responses, with the flash at sample 20.
- `Failures` (348 × 120): stretches of dark record — trials where no photon was absorbed.

Both contain noise generated by the rod in darkness. The `Singles` additionally contain the
single-photon responses themselves, which vary. So the goal is to identify and characterize
the **extra** variability present in `Singles` beyond what is in `Failures`.

In [ ]:
# Time axis: 120 samples at 10 ms, flash at sample index 20 (0-based here, 20 in MATLAB's
# 1-based indexing too -- the original wrote (1:120 - 20)*0.01, so t = 0 falls at index 19).
tme = (np.arange(1, Singles.shape[1] + 1) - 20) * 0.01

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
axes[0].plot(tme, Singles.mean(0),  color=BLUE, lw=1.8, label=f"Singles (n={len(Singles)})")
axes[0].plot(tme, Failures.mean(0), color=RED,  lw=1.8, label=f"Failures (n={len(Failures)})")
axes[0].set(xlabel="time (s)", ylabel="current (pA)", title="mean response")
axes[1].plot(tme, Singles.var(0, ddof=1),  color=BLUE, lw=1.8, label="Singles")
axes[1].plot(tme, Failures.var(0, ddof=1), color=RED,  lw=1.8, label="Failures")
axes[1].set(xlabel="time (s)", ylabel=r"variance (pA$^2$)", title="time-dependent variance")
for ax in axes:
    ax.axvline(0, color="0.5", lw=1, ls="--")
    ax.set_xlim(tme[0], tme[-1])
    ax.legend(fontsize=8)
fig.tight_layout()

print(f"peak of mean single-photon response: {Singles.mean(0).max():.3f} pA "
      f"at t = {tme[Singles.mean(0).argmax()]:.2f} s")

Look at the variance plot. There is extra variance in the singles above what the dark
records explain, and that excess is the variance attributable to fluctuations in the
single-photon response itself.

But the time-dependent variance is **not** a complete description of how the responses vary.
It tells you how much variance is present at each single time point, and nothing about how
correlated the response is at two *different* times. Put another way, knowing the mean and
the time-dependent variance does not let you generate a response — the information about how
to go from a variance back to a signal is missing. It is like being told the square of a
signal and asked to recover the signal: you cannot, because many signals share the same
square. Likewise many different distributions of single-photon responses have the same
time-dependent variance.

PCA can help, because the covariance matrix keeps exactly the information the
time-dependent variance throws away: the off-diagonal terms, i.e. the correlations between
different time points.

In [ ]:
# Covariance of the singles CORRECTED for the covariance of the failures. Both matrices
# use rowvar=False: rows are trials, columns are time points.
CovSingles  = np.cov(Singles,  rowvar=False)
CovFailures = np.cov(Failures, rowvar=False)
SinglesCovar = CovSingles - CovFailures

vmax = np.abs(CovSingles).max()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, Cm, title in [(axes[0], CovSingles,   "cov(Singles)"),
                      (axes[1], CovFailures,  "cov(Failures)"),
                      (axes[2], SinglesCovar, "difference")]:
    im = ax.imshow(Cm, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   extent=[tme[0], tme[-1], tme[-1], tme[0]])
    ax.set(title=title, xlabel="time (s)", ylabel="time (s)")
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.85, label=r"covariance (pA$^2$)")

All three panels share one color scale, so they are directly comparable. The dark noise
(middle) is a smooth band around the diagonal — noise correlated over a few tens of
milliseconds — and is present in the singles too. The difference (right) isolates a blob
concentrated in the first few hundred milliseconds after the flash: the extra, response-related
covariance we are after.

Now the eigenvalues of the corrected covariance.

In [ ]:
SinglesEigVal, SinglesEigVec = pca_eig(SinglesCovar)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
comp = np.arange(1, len(SinglesEigVal) + 1)
axes[0].plot(comp, SinglesEigVal, "o", ms=4, color=BLUE)
axes[0].axhline(0, color="0.5", lw=1)
axes[0].set(xlabel="component", ylabel=r"variance (pA$^2$)",
            title="eigenvalues of cov(Singles) - cov(Failures)")
axes[1].plot(comp[:15], SinglesEigVal[:15], "o-", ms=5, color=BLUE)
axes[1].axhline(0, color="0.5", lw=1)
axes[1].set(xlabel="component", ylabel=r"variance (pA$^2$)",
            title="first 15 components (zoom)", xticks=range(1, 16, 2))
fig.tight_layout()

print("largest eigenvalues:", SinglesEigVal[:8].round(4))
print(f"\n{(SinglesEigVal < 0).sum()} of {len(SinglesEigVal)} eigenvalues are negative; "
      f"most negative = {SinglesEigVal.min():.4f}")

There are a few large eigenvalues and a great many small ones. That is promising: we
probably want to keep the large ones and can hope to discard the small ones without losing
much.

**One honest caveat the MATLAB original passes over.** `SinglesCovar` is a *difference* of
two covariance matrices, so it is not guaranteed to be positive semi-definite, and in fact
many of its eigenvalues come out slightly negative. A genuine covariance matrix cannot have
a negative eigenvalue — a variance cannot be negative — so those are telling you that the
noise subtraction has overshot within sampling error. They are all small compared with the
leading eigenvalues, so they do not affect the leading components, but you should not read
them as "directions with negative variance." They are the level of the noise floor on this
estimate, and a useful eyeball criterion for how many components are worth taking
seriously.

Now let's test the promise directly, by comparing the variance of the singles and the
failures **projected along each eigenvector**. If an axis picks up structure that exists in
the singles but not in the dark records, it is worth keeping. If singles and failures have
similar variance along an axis, that axis is capturing nothing beyond dark noise and we
reject it.

In [ ]:
VarSingles  = (Singles  @ SinglesEigVec).var(0, ddof=1)
VarFailures = (Failures @ SinglesEigVec).var(0, ddof=1)

# The traces were low-pass filtered before they were stored, so they do NOT span all 120
# dimensions -- see the rank check below. Directions outside that subspace carry variance
# ~1e-16, i.e. numerical zero, and the ratio there is meaningless. Mask them out.
alive = VarFailures > 1e-8
ratio = np.where(alive, VarSingles / np.where(alive, VarFailures, 1), np.nan)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax in axes[:2]:
    ax.plot(comp, VarSingles,  "o", ms=5, color=RED,  label="Singles")
    ax.plot(comp, VarFailures, "*", ms=6, color=BLUE, label="Failures")
    ax.set(xlabel="component", ylabel=r"variance of projection (pA$^2$)")
    ax.legend(fontsize=8)
axes[0].set(title="all 120 components (linear)", xlim=(0.5, len(comp) + 0.5))
# Log scale, because on a linear axis everything past component 5 is a flat line at zero
# and you cannot see where the two curves actually meet. Floor the axis at 1e-4 so the
# numerically-empty directions fall off the bottom instead of dominating the plot.
axes[1].set(title="all 120 components (log scale)", yscale="log",
            xlim=(0.5, len(comp) + 0.5), ylim=(1e-4, 1e2))
axes[1].text(0.5, 0.06, f"{(~alive).sum()} directions carry essentially zero variance\n"
             "(data are rank-deficient; see below)", transform=axes[1].transAxes,
             ha="center", fontsize=7.5, color="0.35")
axes[2].plot(comp, ratio, "o", ms=4, color=GREEN)
axes[2].axhline(1, color="0.4", lw=1, ls="--")
axes[2].set(xlabel="component", ylabel="var(Singles) / var(Failures)",
            title="excess variance ratio", xlim=(0.5, len(comp) + 0.5))
fig.tight_layout()

data_rank = np.linalg.matrix_rank(np.vstack([Singles, Failures])
                                  - np.vstack([Singles, Failures]).mean(0))
print(f"numerical rank of the recorded traces: {data_rank} of "
      f"{Singles.shape[1]} time points\n")

print(" comp  var(Singles)  var(Failures)     excess    ratio")
for k in list(range(8)) + [9, 14, 19, 29, 49]:
    print(f"  {k+1:3d}   {VarSingles[k]:10.4f}   {VarFailures[k]:10.4f}   "
          f"{VarSingles[k]-VarFailures[k]:9.4f}   {ratio[k]:6.2f}")
print(f"\nmedian ratio over components 21-120 (excluding the empty directions): "
      f"{np.nanmedian(ratio[20:]):.2f}")

The variance of the projections along the **first few** eigenvectors is much larger for the
singles than for the failures — there is real structure along those axes that the dark
records do not have. Component 1 carries about 3.7× more variance in the singles; by
component 20 the ratio is down to about 1.25, and beyond component 20 the median ratio over
the directions the data actually occupies is 0.81 — at or below 1. Those axes pick up
nothing the dark records do not already have, so we reject them. Note that the ratio decays *slowly* while the absolute excess variance
$\mathrm{var(Singles)} - \mathrm{var(Failures)}$ collapses almost immediately: component 1
alone accounts for about 19.7 pA² of excess, component 5 for 0.26 pA², component 20 for
0.02 pA². It is the absolute excess, not the ratio, that tells you what is worth keeping —
a component can be twice as variable in the singles as in the failures and still contribute
nothing you would notice.

**A second thing the MATLAB original never mentions, and which the log axis makes obvious:
these traces are rank-deficient.** The recorded currents span only a **68-dimensional**
subspace of the 120 time points, because they were low-pass filtered before being stored.
Most of our 120 "components" are therefore directions the data simply never visits, and both
singles and failures project onto them with variance $\sim 10^{-16}$ — numerical zero. Those
are the points that fall off the bottom of the middle panel, and they are why the ratio panel
is blank in the middle: $0/0$ is not a meaningful ratio. (The exact count of empty directions
depends on where you put the threshold: `matrix_rank` with its default tolerance says 68
occupied, hence 52 empty, while the cruder cutoff used to mask the plot — projected variance
below $10^{-8}$ pA² — flags 63. The point is not the exact number but that a large majority
of the nominal dimensions are unoccupied.)

This also explains the negative eigenvalues. The 120 sorted eigenvalues split into roughly
25 clearly positive ones, then a block of numerical zeros from the null space, then the
negative tail. The rightmost components in the ratio panel sit *below* 1 — the dark records are more
variable than the singles along those directions — which is a sampling-error effect of
estimating a 120×120 covariance from 236 and 348 traces. A good habit whenever you build a
covariance matrix from limited data: check `np.linalg.matrix_rank`, and remember that you
cannot estimate more independent directions than you have samples.

(In the MATLAB file this same plot is discussed in terms of the *last* few components and
zoomed with `xlim([110 120])`, because `eig` returned the eigenvalues smallest-first. We
sorted descending, so the interesting components are the first few. Same components,
opposite end of the axis.)

Most of the action is in the first two or three components. Let's take three, and reduce
each response to a single point in a 3-D space whose axes are those eigenvectors, with each
response specified by its projections along them.

First, look at the three eigenvectors themselves.

In [ ]:
n_comp = 3
EigVec = SinglesEigVec[:, :n_comp]        # the 3 axes we will use
EigVal = SinglesEigVal[:n_comp]

fig, ax = plt.subplots(figsize=(8, 4.2))
for k, color in zip(range(n_comp), [BLUE, RED, GREEN]):
    ax.plot(tme, EigVec[:, k], lw=1.8, color=color,
            label=f"component {k+1} ($\\lambda$ = {EigVal[k]:.2f} pA$^2$)")
ax.plot(tme, Singles.mean(0) / np.linalg.norm(Singles.mean(0)), "--", color="0.4", lw=1.5,
        label="mean single-photon response (normalized)")
ax.axvline(0, color="0.5", lw=1, ls=":")
ax.axhline(0, color="0.5", lw=0.8)
ax.set(xlabel="time (s)", ylabel="weight (arbitrary units)",
       title="leading principal components of the rod single-photon response",
       xlim=(tme[0], tme[-1]))
ax.legend(fontsize=8)
fig.tight_layout()

mean_hat = Singles.mean(0) / np.linalg.norm(Singles.mean(0))
print("overlap |PC_k . mean_hat|:",
      np.round(np.abs(EigVec.T @ mean_hat), 3), " (1.0 would mean identical shape)")
print("pre-flash rms as a fraction of full rms, per component:",
      np.round([EigVec[:19, k].std() / EigVec[:, k].std() for k in range(n_comp)], 3))

They look reasonable. Components 1 and 2 have little structure in the first 20 samples,
which precede the flash — the pre-flash rms is only 7% and 10% of each component's overall
rms. That is a good sanity check, since nothing before the flash should carry
response-related variance. Component 3 is noisier (28%), a hint that it is already close to
the noise floor, consistent with its small eigenvalue.

Note, though, that **none of them looks like the mean single-photon response** (dashed grey).
Component 1 has the largest overlap with the mean (0.75, where 1.0 would mean an identical
shape), but it is plainly a different waveform: it rises late and peaks around 0.55 s, where
the mean peaks at 0.26 s. Component 2 tracks the rise and peak of the mean more closely
(overlap 0.61) but then swings negative, and component 3 overlaps the mean hardly at all
(0.22).

Why doesn't a component just come out looking like the mean? That is homework question 2(d),
and it is the single most commonly misunderstood point about PCA: `np.cov` subtracts the mean
before it ever computes anything, so the components describe the directions of greatest
**variability about the mean**, not the mean itself. Component 1 says *responses that are
large late are large late* — a timing/duration axis — not *this is what a response looks
like*.

Now describe individual responses by their projections along these three axes — that is, by
the weight of each component. Here are three real responses and their three-component
descriptions.

In [ ]:
start = 10                       # first response to look at (MATLAB used the same offset)
sel = np.arange(start + 1, start + 4)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, idx in zip(axes, sel):
    coef = Singles[idx] @ EigVec                     # projection onto each component
    recon = EigVec @ coef                            # sum of weighted components
    ax.plot(tme, Singles[idx], color=BLUE, lw=1.2, label="response")
    ax.plot(tme, recon, color=RED, lw=1.8, label="3-component fit")
    ax.axvline(0, color="0.5", lw=1, ls=":")
    ax.set(xlabel="time (s)", title=f"response {idx}\ncoefs = " +
           ", ".join(f"{c:.2f}" for c in coef))
axes[0].set_ylabel("current (pA)")
axes[0].legend(fontsize=8)
fig.tight_layout()

So we are expanding each response in a series, with the basis set given by the eigenvectors
and the expansion coefficients given by the projections of the response onto them:

$$r(t) \;\approx\; \sum_{k=1}^{3}\; c_k\, v_k(t), \qquad
c_k \;=\; \sum_t r(t)\, v_k(t)$$

Blue is the recorded response, red the three-component reconstruction. There is also dark
noise in the recorded responses, so we should not expect perfect agreement — and the
mismatch is mostly fast wiggle, i.e. exactly the dark noise we deliberately excluded.

### Verifying our from-scratch PCA against `sklearn`

We built the components by hand from the covariance eigensystem, because that mechanism *is*
the lesson. It is still worth checking against a standard implementation. Two mismatches to
watch for:

- `sklearn.decomposition.PCA` **always subtracts the mean** and works from
  $\mathrm{cov}(X)$ alone. It cannot do the noise-corrected $\mathrm{cov(Singles)} -
  \mathrm{cov(Failures)}$ that this tutorial uses, so we compare against a plain PCA of
  `Singles`.
- sklearn computes the components via the **SVD**, and `np.linalg.svd` returns
  $V^{\mathsf T}$ where MATLAB's `[U,S,V] = svd(X)` returns $V$. sklearn's
  `pca.components_` follows the NumPy convention: it is $V^{\mathsf T}$, i.e. components are
  the **rows**, not the columns. Our `EigVec` has components in the columns.
- Signs are arbitrary in both, so compare $|\cdot|$ or align signs first.

In [ ]:
w_plain, V_plain = pca_eig(np.cov(Singles, rowvar=False))     # our own, no noise correction

pca = PCA(n_components=5).fit(Singles)
V_sk = pca.components_.T          # transpose: sklearn stores components as ROWS (V^T)

# Align signs before comparing -- eigenvector sign is arbitrary in both implementations.
sgn = np.sign(np.sum(V_plain[:, :5] * V_sk, axis=0))
V_sk = V_sk * sgn

print("eigenvalues, ours   :", w_plain[:5].round(5))
print("explained_variance_ :", pca.explained_variance_[:5].round(5))
print("max |difference|    :", np.abs(w_plain[:5] - pca.explained_variance_[:5]).max())
print()
print("max |difference| between eigenvectors (after sign alignment):",
      np.abs(V_plain[:, :5] - V_sk).max())
print()
print("How much does the noise correction change the leading component?")
print("  |dot(plain PC1, noise-corrected PC1)| =",
      round(abs(V_plain[:, 0] @ SinglesEigVec[:, 0]), 5))

The agreement is to numerical precision, so our from-scratch covariance eigensystem *is*
PCA. Note also that the leading component barely changes when we drop the noise correction —
the single-photon signal dominates that direction. The correction matters much more for the
lower components, where signal and dark noise are comparable.

### The distribution of singles and failures in the 3-D component space

Each response is now a single point in a 3-D space. Because a static notebook cannot be
rotated with the mouse the way `rotate3d on` allowed, the figure below shows the 3-D scatter
alongside all three pairwise 2-D projections.

In [ ]:
PS = Singles  @ EigVec       # (236, 3) projections of the singles
PF = Failures @ EigVec       # (348, 3) projections of the failures

fig = plt.figure(figsize=(12, 7.5))
ax3 = fig.add_subplot(2, 2, 1, projection="3d")
ax3.scatter(*PS.T, s=14, color=RED,  alpha=0.75, label="Singles",  depthshade=False)
ax3.scatter(*PF.T, s=14, color=BLUE, alpha=0.55, marker="*", label="Failures",
            depthshade=False)
ax3.set(xlabel="component 1", ylabel="component 2", zlabel="component 3")
ax3.set_title("3-D component space")
ax3.view_init(elev=20, azim=-58)
ax3.legend(fontsize=8, loc="upper left")

# One shared window for all three 2-D panels (same range on every axis, equal aspect),
# so the spreads really are comparable by eye rather than rescaled panel by panel.
allp = np.vstack([PS, PF])
pad = 0.06 * (allp.max() - allp.min())
lo, hi = allp.min() - pad, allp.max() + pad
for k, (i, j) in enumerate([(0, 1), (0, 2), (1, 2)]):
    ax = fig.add_subplot(2, 2, k + 2)
    ax.plot(PF[:, i], PF[:, j], "*", ms=4, color=BLUE, alpha=0.55, label="Failures")
    ax.plot(PS[:, i], PS[:, j], "o", ms=4, color=RED,  alpha=0.75, label="Singles")
    ax.set(xlabel=f"component {i+1}", ylabel=f"component {j+1}",
           xlim=(lo, hi), ylim=(lo, hi), aspect="equal")
    if k == 0:
        ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()

for name, P in [("Singles ", PS), ("Failures", PF)]:
    print(f"{name}  mean = {np.round(P.mean(0), 2)}   sd = {np.round(P.std(0, ddof=1), 2)}")

Two things should be visible.

**First, the means of the two distributions differ.** The singles are centred near
$(10.3,\ 8.4,\ 3.0)$ while the failures sit at $(-0.1,\ 0.3,\ 0.1)$ — essentially the origin,
as they must, since a dark record projected onto response-shaped axes has no systematic
projection. The separation is largest along components 1 and 2. That is good: there is a lot
of structure in the mean single-photon response and we want it captured.

**Second, the singles distribution is more spread out than the failures.** The standard
deviations are $(5.2,\ 2.5,\ 1.8)$ for the singles against $(2.7,\ 1.6,\ 1.3)$ for the
failures — about 1.9×, 1.6× and 1.4× along components 1, 2 and 3 — so the excess spread is
clearest along components 1 and 2 and weakest along 3, matching the variance table above. That extra spread *is* the trial-to-trial variability of the single-photon
response, now expressed as a cloud in three dimensions instead of a 120-dimensional mess.

Finally, how much of the variability did we actually keep?

In [ ]:
Sc = Singles - Singles.mean(0)                  # variance is about the mean
total_var = (Sc ** 2).sum() / (len(Singles) - 1)
var_per_comp = ((Sc @ SinglesEigVec) ** 2).sum(0) / (len(Singles) - 1)
frac = np.cumsum(var_per_comp) / total_var

fig, ax = plt.subplots(figsize=(7, 4.0))
ax.plot(comp, 100 * frac, "o-", ms=4, color=BLUE)
for k in (3, 10):
    ax.axvline(k, color=GREY, ls="--", lw=1)
    ax.annotate(f"{100*frac[k-1]:.0f}% with {k}", xy=(k, 100 * frac[k - 1]),
                xytext=(k + 18, 100 * frac[k - 1] - 12), fontsize=9,
                arrowprops=dict(arrowstyle="->", color=GREY))
ax.set(xlabel="number of components kept", ylabel="cumulative % of Singles variance",
       title="variance of the singles captured by the leading components",
       xlim=(0, 121), ylim=(0, 102))
fig.tight_layout()

for k in (1, 2, 3, 5, 10, 20, 40, 68, 120):
    print(f"{k:3d} components -> {100 * frac[k-1]:5.1f}% of the variance of Singles")

Three components capture **82%** of the variance of the singles and ten capture **91%** — so
we have gone from 120 numbers per response to 3 while keeping most of the structure.

The curve does something odd worth understanding: it climbs steeply, plateaus near 95% from
about component 25 to component 95, then climbs again to 100% at the very end. That is a
direct consequence of the noise correction. The axes are ordered by the eigenvalues of
$\mathrm{cov(Singles)} - \mathrm{cov(Failures)}$, not of $\mathrm{cov(Singles)}$, so the
middle block is the empty null-space directions (they add nothing) and the final block is the
negative-eigenvalue directions, which do still contain a few percent of the singles' variance.
Order by the eigenvalues of `np.cov(Singles, rowvar=False)` instead and the curve is
monotone-concave, as a scree plot normally is — but the leading components are then
contaminated by dark noise. Pick your poison knowingly.

What has all this bought us? We have described a set of single-photon responses in a
relatively low-dimensional space — three numbers per response instead of 120. That helps in
two ways.

First, we can now describe the *nature* of the variations about the mean response and compare
them against competing models for how that variability is generated (does the rod vary in
amplitude? in timing? in shutoff kinetics?). Second, we can use the fitted distribution of
coefficients to build a **generative model** that produces synthetic responses with
statistics close to those of the real ones — draw three coefficients, weight the three
components, add dark noise.

> ### Homework question 2
> **(a)** Can you think of another case in your own work where dimensional reduction is
> important?
>
> **(b)** How much of the variance of the singles are we capturing with the first 3
> components above? How much more do we get with 10? (The last cell prints both — make sure
> you can say *why* the curve has the shape it does, and why it does not start near 100%.)
>
> **(c)** Why do we subtract the covariance of the failures from that of the singles? What
> would the leading components look like if we did not? Try it: rerun the analysis with
> `SinglesCovar = np.cov(Singles, rowvar=False)` and compare.
>
> **(d)** Why doesn't any of the eigenvectors we recover look like the mean single-photon
> response? *Hint:* what did the covariance matrix subtract off before we ever took its
> eigenvectors?
>
> **(e)** Several eigenvalues of the noise-corrected covariance come out negative. What does
> that mean, and how could you use their size to decide how many components to keep?
>
> **(f)** The traces span only a 68-dimensional subspace of the 120 time points. Where did
> the other 52 dimensions go, and what would change if the data had been stored unfiltered?

---
## Summary

1. **Dimensional reduction** is unavoidable. Every choice of filter, bin width, or sampling
   rate is a choice of which dimensions of the data to keep. PCA makes that choice
   systematically instead of by eye.

2. The **covariance matrix** holds the variance of each variable on the diagonal and the
   covariance between variables off it. Unlike the time-dependent variance alone, it records
   how the signal at one time relates to the signal at another.

3. The **eigenvectors of the covariance matrix** form a coordinate system in which the
   covariance is diagonal — the projections onto different eigenvectors are uncorrelated —
   and the **eigenvalue is the variance along its eigenvector**. Because the eigenvalues sum
   to the trace, i.e. the total variance, this ranks the axes and tells you exactly what
   fraction of the variance any subset captures.

4. Applied to rod photoreceptors, PCA reduces each 120-point single-photon response to
   **three numbers**, retaining most of the trial-to-trial structure. Comparing the
   projections of the singles against the dark records tells us which components carry real
   signal and which are dark noise.

5. Two habits worth keeping: subtract a **noise covariance** estimated from control trials
   whenever you have one, and always check *how much* variance your retained components
   capture rather than assuming a small number is enough.

### MATLAB → Python cheat sheet for this tutorial

| MATLAB | Python | trap |
|---|---|---|
| `cov(X)` | `np.cov(X, rowvar=False)` | NumPy's default treats **rows** as variables — silently wrong on square input |
| `corrcoef(X)` | `np.corrcoef(X, rowvar=False)` | same |
| `[V,D] = eig(C)` | `w, V = np.linalg.eigh(C)` | `eigh` for symmetric matrices; returns eigenvalues **ascending**, so sort |
| `eigs(C, 3)` | sort `eigh` output, take first 3 | `eigs` returns *largest* by default |
| `C^0.5` | `scipy.linalg.sqrtm(C)` | `C**0.5` in NumPy is **elementwise** |
| `[U,S,V] = svd(X)` | `U, s, Vt = np.linalg.svd(X)` | NumPy returns $V^{\mathsf T}$; sklearn's `components_` is also $V^{\mathsf T}$ |
| `var(x)` | `np.var(x, ddof=1)` | MATLAB normalizes by $N-1$, NumPy by $N$ |
| `Dist * EigVec` | `Dist @ EigVec` | `*` is elementwise in NumPy |
| (any eigenvector) | (any eigenvector) | **sign is arbitrary** — figures may come out mirrored |

### Further reading

- Rieke & Baylor (1998). Origin of reproducibility in the responses of retinal rods to
  single photons. *Biophysical Journal* **75**, 1836–1857. — the experiment behind
  `RodData.mat`.
- Schwartz, Pillow, Rust & Simoncelli (2006). Spike-triggered neural characterization.
  *Journal of Vision* **6**, 484–507. — spike-triggered covariance, i.e. this same
  eigen-machinery applied to receptive fields.
- Cunningham & Yu (2014). Dimensionality reduction for large-scale neural recordings.
  *Nature Neuroscience* **17**, 1500–1509.
- [`PCANeuroPopTutorial.ipynb`](PCANeuroPopTutorial.ipynb) — the same tools applied to the
  geometry of neural population activity.
- [`ClassificationTutorial.ipynb`](ClassificationTutorial.ipynb) — why the highest-variance
  direction is not always the most useful one.